Librerías utilizadas

In [1]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec
import time
from transformers import AutoModel
import os
import unicodedata
load_dotenv()

/home/rodri/Documents/Maestria/proyectos_maestria/procesamiento_lenguaje_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Se leen api keys de pinecone y groq

In [2]:
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
groq_api_key = os.environ.get("GROQ_API_KEY")

Se carga el modelo de embeddings en español/inglés de Jina. Se utiliza este modelo debido a que tenemos dos cvs en español y uno en inglés

In [3]:
model = AutoModel.from_pretrained('jinaai/jina-embeddings-v2-base-es', trust_remote_code=True) # trust_remote_code is needed to use the encode method

Se arma el cliente de Pinecone y define el nombre del índice que voy a usar.

In [4]:
pc = Pinecone(api_key=pinecone_api_key)

spec = ServerlessSpec(cloud="aws", region="us-east-1")

# choose a name for your index
index_name = "cv-search"

Si el índice no existe se crea

In [5]:
# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=768,
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '154',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 30 Nov 2025 21:08:19 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '32',
                                    'x-pinecone-request-id': '8280139971297863159',
                                    'x-pinecone-request-latency-ms': '34'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

Procesamiento de los CVs en Markdown. Se divide al markdown en base a sus secciones,y para cada sección se genera un nuevo vector. Además, se agrega metadata para intentar mejorar los resultados

In [6]:
cv_list = os.listdir("cvs_markdown")

Función rápida para partir el markdown por secciones h2 y quedarme con título y contenido.

In [7]:
from markdown_it import MarkdownIt

def parse_md_sections(md_text):
    md = MarkdownIt()
    tokens = md.parse(md_text)

    sections = []
    current = None

    for tok in tokens:
        if tok.type == "heading_open" and tok.tag == "h2":
            # empezar nueva sección
            if current:
                sections.append(current)
            current = {"title": None, "content": ""}

        elif tok.type == "inline" and current and current["title"] is None:
            current["title"] = tok.content

        elif tok.type == "paragraph_open":
            pass

        elif tok.type == "inline" and current:
            current["content"] += tok.content + "\n"

    if current:
        sections.append(current)

    return sections

Se recorre cada archivo y guarda las secciones asociadas al nombre del CV en una lista.

In [8]:
section_list = []

for file in cv_list:
    with open(os.path.join(".","cvs_markdown",file)) as f:
        md = f.read()
    sections = parse_md_sections(md)
    section_list.append({"cv_name":file.split(".")[0], "sections":sections})

Se arman los vectores: normalizando nombres/títulos. Se generan IDs y ubtienen los embeddings de cada sección.

In [9]:
vectors = []

for section in section_list:
    name = unicodedata.normalize('NFKD', section["cv_name"]).encode('ascii', 'ignore').decode('ascii')
    content = section["sections"]

    for part in content:
        title = unicodedata.normalize('NFKD', part["title"]).encode('ascii', 'ignore').decode('ascii')
        id = f"{name}_{title}"
        embeddings = model.encode(part["content"])
        embeddings = embeddings.tolist() if hasattr(embeddings, "tolist") else embeddings
        vectors.append({
            "id": id,
            "values": embeddings,
            "metadata": {
                "nombre_cv": name,
                "seccion": part["title"],
                "content": part["content"],
            },
        })


Se suben todos los vectores a Pinecone dentro del namespace elegido.

In [10]:
index.upsert(
    vectors=vectors,
    namespace="vs_namespace"
    )

UpsertResponse(upserted_count=21, _response_info={'raw_headers': {'date': 'Sun, 30 Nov 2025 21:08:27 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '72904', 'x-pinecone-request-latency-ms': '2883', 'x-pinecone-request-id': '9128910301299948580', 'x-envoy-upstream-service-time': '1582', 'grpc-status': '0', 'server': 'envoy'}})